Products raw file

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

data = [
    Row(product_id=501, product_name="Laptop Pro 14", category="Electronics", brand="Dell", price=85000, created_date="01-11-2024", modified_date="05-11-2024"),
    Row(product_id=502, product_name="Gaming Mouse", category="Electronics", brand="Logitech", price=2500, created_date="03-11-2024", modified_date="10-11-2024"),
    Row(product_id=503, product_name="Mechanical Keyboard", category="Electronics", brand="Keychron", price=6500, created_date="05-11-2024", modified_date="15-11-2024"),
    Row(product_id=504, product_name="Office Chair", category="Furniture", brand="GreenSoul", price=12000, created_date="07-11-2024", modified_date="20-11-2024"),
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=15000, created_date="09-11-2024", modified_date="25-11-2024"),
    Row(product_id=506, product_name="Water Bottle", category="Lifestyle", brand="Milton", price=500, created_date="11-11-2024", modified_date="18-11-2024"),
    Row(product_id=507, product_name="Wireless Earbuds", category="Electronics", brand="Boat", price=3500, created_date="13-11-2024", modified_date="22-11-2024"),
    Row(product_id=508, product_name="Smart Watch", category="Electronics", brand="Noise", price=5500, created_date="15-11-2024", modified_date="28-11-2024"),
    Row(product_id=509, product_name="Backpack", category="Lifestyle", brand="Skybags", price=2200, created_date="17-11-2024", modified_date="30-11-2024"),
    Row(product_id=510, product_name="Monitor 27 Inch", category="Electronics", brand="LG", price=22000, created_date="19-11-2024", modified_date="21-11-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

products_raw = spark.createDataFrame(data, schema) 

products_raw = products_raw \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

display(products_raw)

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,22000,2024-11-19T00:00:00.000Z,2024-11-21T00:00:00.000Z


In [0]:
# spark.sql("DROP TABLE IF EXISTS catalog_project1.source1.products_raw")
products_raw.write.mode("append")\
                  .format("delta")\
                  .option("mergeSchema", "true")\
                  .saveAsTable("catalog_project1.source1.products_raw")

In [0]:
spark.sql("""
SELECT * 
FROM catalog_project1.source1.products_raw 
WHERE to_date(modified_date, 'dd-MM-yyyy') < to_date(created_date, 'dd-MM-yyyy')
""").display()

product_id,product_name,category,brand,price,created_date,modified_date


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=510, product_name="Monitor 27 Inch", category="Electronics", brand="LG", price=24000, created_date="19-11-2024", modified_date="01-12-2024"),
    Row(product_id=511, product_name="Monitor 30 Inch", category="Electronics", brand="LG", price=30000, created_date="01-12-2024", modified_date="01-12-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated, 1 record inserted")

Incremental load completed: 1 record updated, 1 record inserted


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,24000,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=511, product_name="Monitor 32 Inch", category="Electronics", brand="LG", price=45000, created_date="01-12-2024", modified_date="02-12-2024"), 
    Row(product_id=512, product_name="Monitor 21 Inch", category="Electronics", brand="LG", price=23000, created_date="02-12-2024", modified_date="02-12-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated, 1 record inserted")

Incremental load completed: 1 record updated, 1 record inserted


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,24000,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=512, product_name="Monitor 23 Inch", category="Electronics", brand="LG", price=25200, created_date="02-12-2024", modified_date="03-12-2024"), 
    Row(product_id=513, product_name="Backpack", category="Lifestyle", brand="SkyHighbags", price=2199, created_date="03-12-2024", modified_date="03-12-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated, 1 record inserted")

Incremental load completed: 1 record updated, 1 record inserted


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
513,Backpack,Lifestyle,SkyHighbags,2199,2024-12-03T00:00:00.000Z,2024-12-03T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=513, product_name="Waterproof Backpack", category="Lifestyle", brand="SkyHighbags", price=2300, created_date="03-12-2024", modified_date="04-12-2024"), 
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=999, created_date="04-12-2024", modified_date="04-12-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated, 1 record inserted")

Incremental load completed: 1 record updated, 1 record inserted


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
513,Waterproof Backpack,Lifestyle,SkyHighbags,2300,2024-12-03T00:00:00.000Z,2024-12-04T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=1499, created_date="05-12-2024", modified_date="05-12-2024"), 
    Row(product_id=515, product_name="High Quality Backpack", category="Lifestyle", brand="SkyHighbags", price=1999, created_date="05-12-2024", modified_date="05-12-2024")
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated, 1 record inserted")

Incremental load completed: 1 record updated, 1 record inserted


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
515,High Quality Backpack,Lifestyle,SkyHighbags,1999,2024-12-05T00:00:00.000Z,2024-12-05T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=1299, created_date="05-12-2024", modified_date="06-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
515,High Quality Backpack,Lifestyle,SkyHighbags,1999,2024-12-05T00:00:00.000Z,2024-12-05T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=99, created_date="05-12-2024", modified_date="07-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
515,High Quality Backpack,Lifestyle,SkyHighbags,1999,2024-12-05T00:00:00.000Z,2024-12-05T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=9, created_date="05-12-2024", modified_date="08-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
515,High Quality Backpack,Lifestyle,SkyHighbags,1999,2024-12-05T00:00:00.000Z,2024-12-05T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=514, product_name="Everyday Backpack", category="Lifestyle", brand="SkyHighbags", price=0, created_date="05-12-2024", modified_date="09-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
505,Study Table,Furniture,IKEA,15000,2024-11-09T00:00:00.000Z,2024-11-25T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
515,High Quality Backpack,Lifestyle,SkyHighbags,1999,2024-12-05T00:00:00.000Z,2024-12-05T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=10000, created_date="09-11-2024", modified_date="10-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,24000,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z
511,Monitor 32 Inch,Electronics,LG,45000,2024-12-01T00:00:00.000Z,2024-12-02T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=9999, created_date="09-11-2024", modified_date="11-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=999, created_date="09-11-2024", modified_date="12-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=9099, created_date="09-11-2024", modified_date="13-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=99, created_date="09-11-2024", modified_date="14-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=9, created_date="09-11-2024", modified_date="15-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,24000,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z
511,Monitor 32 Inch,Electronics,LG,45000,2024-12-01T00:00:00.000Z,2024-12-02T00:00:00.000Z


In [0]:
from pyspark.sql import Row
from pyspark.sql.types import *
from pyspark.sql.functions import to_timestamp 

# Incremental data with 1 update and 1 new record
data = [
    Row(product_id=505, product_name="Study Table", category="Furniture", brand="IKEA", price=15000, created_date="09-11-2024", modified_date="17-12-2024") 
]

schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", IntegerType(), True),
    StructField("created_date", StringType(), True),
    StructField("modified_date", StringType(), True)
])

# Create DataFrame with same schema
products_incremental = spark.createDataFrame(data, schema)

# Convert date strings to timestamps
products_incremental = products_incremental \
    .withColumn("created_date", to_timestamp("created_date", "dd-MM-yyyy")) \
    .withColumn("modified_date", to_timestamp("modified_date", "dd-MM-yyyy"))

# Use Delta merge for upsert (update existing + insert new)
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "catalog_project1.source1.products_raw")

delta_table.alias("target").merge(
    products_incremental.alias("source"),
    "target.product_id = source.product_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

print("Incremental load completed: 1 record updated")

Incremental load completed: 1 record updated


In [0]:
spark.sql("SELECT * FROM catalog_project1.source1.products_raw").display()

product_id,product_name,category,brand,price,created_date,modified_date
501,Laptop Pro 14,Electronics,Dell,85000,2024-11-01T00:00:00.000Z,2024-11-05T00:00:00.000Z
502,Gaming Mouse,Electronics,Logitech,2500,2024-11-03T00:00:00.000Z,2024-11-10T00:00:00.000Z
503,Mechanical Keyboard,Electronics,Keychron,6500,2024-11-05T00:00:00.000Z,2024-11-15T00:00:00.000Z
504,Office Chair,Furniture,GreenSoul,12000,2024-11-07T00:00:00.000Z,2024-11-20T00:00:00.000Z
506,Water Bottle,Lifestyle,Milton,500,2024-11-11T00:00:00.000Z,2024-11-18T00:00:00.000Z
507,Wireless Earbuds,Electronics,Boat,3500,2024-11-13T00:00:00.000Z,2024-11-22T00:00:00.000Z
508,Smart Watch,Electronics,Noise,5500,2024-11-15T00:00:00.000Z,2024-11-28T00:00:00.000Z
509,Backpack,Lifestyle,Skybags,2200,2024-11-17T00:00:00.000Z,2024-11-30T00:00:00.000Z
510,Monitor 27 Inch,Electronics,LG,24000,2024-11-19T00:00:00.000Z,2024-12-01T00:00:00.000Z
511,Monitor 32 Inch,Electronics,LG,45000,2024-12-01T00:00:00.000Z,2024-12-02T00:00:00.000Z
